# LTB 수요예측 운영 Notebook

부품의 **직전 24개월 수요**로 LTB 결정월부터 **향후 60개월의 월별 수요량**을 예측합니다. 결정월 전에 관측이 끝난 부품으로 학습하고, 예측 대상의 과거 수요만으로 월별 패턴과 총량을 계산합니다.

실행 순서: **설정 → 입력 확인 → 학습·예측 → 결과 확인·저장**. 이 Notebook은 로컬 배치용입니다. DB에 접속하지 않으며, 산출값은 발주량이 아닌 수요 추정치입니다. 실제 운영 데이터 검증과 적용 승인은 별도입니다.

## 1. 실행 설정

`ts_forecaster` 커널을 선택합니다. 아래 셀의 설정값 3개를 직접 입력하거나 이름 앞에 `LTB_OP_`를 붙인 환경변수로 지정한 뒤 **Run All**을 실행합니다. 결과 저장은 선택 사항입니다.

- `DECISION_MONTH`: 예측 시작월의 첫날. 예: `2025-03-01`
- `TRAIN_PATH`, `SCORING_PATH`: 학습용·예측용 Parquet 파일의 로컬 절대 경로
- `OUTPUT_PATH`: 저장할 새 Parquet 파일의 절대 경로. 생략하면 Notebook의 `forecasts` 변수에만 보관
- `K`: 기본값 3. 한 군집에 최소 20개 학습 부품이 필요하며, 기준을 못 채우면 실행 중단

In [ ]:
import os
from pathlib import Path

DECISION_MONTH = os.getenv("LTB_OP_DECISION_MONTH")
TRAIN_PATH = os.getenv("LTB_OP_TRAIN_PATH")
SCORING_PATH = os.getenv("LTB_OP_SCORING_PATH")
OUTPUT_PATH = os.getenv("LTB_OP_OUTPUT_PATH")

K = 3
MIN_CLUSTER_PARTS = 20
SEED = 20261124

required = (DECISION_MONTH, TRAIN_PATH, SCORING_PATH)
if not all(required) and (any(required) or OUTPUT_PATH):
    raise ValueError("DECISION_MONTH, TRAIN_PATH, SCORING_PATH을 모두 설정하세요.")
run_enabled = all(required)
if not run_enabled:
    print("설정값 3개를 입력한 뒤 Run All을 실행하세요. 데이터는 읽지 않았습니다.")


## 2. 입력 데이터 확인

두 파일 모두 **부품 × 월**당 한 행을 갖습니다. `part_key`는 법인별로 같은 부품번호를 구분할 수 있는 고유 키입니다. 모든 월 컬럼은 매월 1일을 나타내는 Date 또는 `YYYY-MM-DD` 문자열, `demand_qty`는 0 이상의 수치여야 합니다.

| 입력 | 필수 컬럼 | 포함 기간 |
| --- | --- | --- |
| 학습 | `part_key`, `decision_month`, `completed_month`, `month`, `demand_qty` | 부품별 과거 결정월 직전 24개월 + 결정월부터 실적 60개월 |
| 예측 | `part_key`, `month`, `demand_qty` | 이번 결정월 직전 24개월 |

`completed_month`는 학습 부품의 관측 종료월입니다. 실적 60개월 전체가 끝났고 관측 종료월이 이번 결정월보다 앞선 부품만 학습에 사용합니다. 예측 대상과 학습 부품이 겹치면 중단합니다. **관측된 무수요만 0으로 입력**합니다. 누락·중복·기간 밖 행은 자동으로 채우거나 합치지 않습니다.

예를 들어 2025년 3월 결정분의 예측 입력은 2023년 3월~2025년 2월이고, 예측 결과는 2025년 3월~2030년 2월입니다.

In [ ]:
import re
from datetime import date

import numpy as np
import polars as pl


def _parse_run_month(value):
    if isinstance(value, str) and re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", value):
        value = date.fromisoformat(value)
    if type(value) is not date or value.day != 1:
        raise ValueError("decision_month must be a month-start Date or YYYY-MM-DD")
    return value


def _select_ltb_columns(frame, columns, label):
    if not isinstance(frame, pl.DataFrame):
        raise TypeError(f"{label} must be a Polars DataFrame")
    missing = set(columns).difference(frame.columns)
    if missing:
        raise ValueError(f"{label} missing columns: {sorted(missing)}")
    if not frame.height:
        raise ValueError(f"{label} contains no rows")
    if frame.schema["part_key"] != pl.String or frame.select(
        (pl.col("part_key").is_null() | (pl.col("part_key").str.strip_chars() == "")).any()
    ).item():
        raise ValueError(f"{label}.part_key must contain nonempty strings")
    return frame.select(columns)


def _ltb_month_index(frame, column, label):
    dtype = frame.schema[column]
    value = pl.col(column)
    if dtype == pl.String:
        value = pl.when(value.str.contains(r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$")).then(
            value.str.to_date("%Y-%m-%d", strict=False, exact=True)
        ).otherwise(None)
    elif dtype != pl.Date:
        raise TypeError(f"{label}.{column} must be Date or YYYY-MM-DD string")
    month_index = pl.when(value.dt.day() == 1).then(
        value.dt.year() * 12 + value.dt.month().cast(pl.Int32) - 1
    ).otherwise(None)
    frame = frame.with_columns(month_index.alias(column))
    if frame[column].null_count():
        raise ValueError(f"{label}.{column} must contain valid month-start dates")
    return frame


def _ltb_qty(frame, label):
    if not frame.schema["demand_qty"].is_numeric():
        raise TypeError(f"{label}.demand_qty must be numeric")
    frame = frame.with_columns(pl.col("demand_qty").cast(pl.Float64))
    values = pl.col("demand_qty")
    if frame.select((values.is_null() | ~values.is_finite() | (values < 0)).any()).item():
        raise ValueError(f"{label}.demand_qty must contain finite nonnegative values")
    return frame


def _ltb_month_matrix(frame, decision_index, first, last, label):
    width = last - first + 1
    frame = frame.with_columns((pl.col("month") - decision_index).alias("_offset"))
    checked = frame.group_by("part_key").agg(
        pl.len().alias("_rows"),
        pl.col("_offset").n_unique().alias("_unique"),
        pl.col("_offset").min().alias("_first"),
        pl.col("_offset").max().alias("_last"),
    )
    invalid = checked.filter(
        (pl.col("_rows") != width)
        | (pl.col("_unique") != width)
        | (pl.col("_first") != first)
        | (pl.col("_last") != last)
    )
    if invalid.height:
        raise ValueError(
            f"{label}: {invalid.height} part(s) do not contain exactly one row per month "
            f"at offsets {first}..{last}; check missing, duplicate and out-of-window months"
        )
    values = frame.sort("part_key", "_offset")["demand_qty"].to_numpy()
    return values.reshape(checked.height, width)


def prepare_ltb_operating_inputs(train_frame, scoring_frame, decision_month):
    """Validate long-form monthly data and build aligned arrays for one LTB run.

    A training part contributes only if its completion month is before this run
    and the complete 60-month outcome period is already covered. Keep observed
    zero-demand rows explicit; a missing row is never treated as zero demand.
    """
    decision_month = _parse_run_month(decision_month)
    run_index = decision_month.year * 12 + decision_month.month - 1
    train = _select_ltb_columns(
        train_frame, ["part_key", "decision_month", "completed_month", "month", "demand_qty"], "train"
    )
    scoring = _select_ltb_columns(scoring_frame, ["part_key", "month", "demand_qty"], "scoring")

    for column in ("decision_month", "completed_month"):
        train = _ltb_month_index(train, column, "train")
    cohort = train.group_by("part_key").agg(
        pl.col("decision_month").first(),
        pl.col("completed_month").first(),
        pl.col("decision_month").n_unique().alias("_decisions"),
        pl.col("completed_month").n_unique().alias("_completions"),
    )
    if cohort.filter((pl.col("_decisions") != 1) | (pl.col("_completions") != 1)).height:
        raise ValueError("train: each part_key needs one historical decision_month and completed_month")
    eligible = cohort.filter(
        (pl.col("completed_month") < run_index)
        & (pl.col("completed_month") >= pl.col("decision_month") + 59)
    ).select("part_key")
    if not eligible.height:
        raise ValueError("train: no completed parts with a full 60-month outcome before decision_month")
    if eligible.join(scoring.select("part_key").unique(), on="part_key", how="semi").height:
        raise ValueError("An eligible training part_key is also present in scoring")

    train = train.join(eligible, on="part_key", how="semi")
    train = _ltb_qty(_ltb_month_index(train, "month", "train"), "train")
    train_matrix = _ltb_month_matrix(train, pl.col("decision_month"), -24, 59, "train")
    scoring = _ltb_qty(_ltb_month_index(scoring, "month", "scoring"), "scoring")
    scoring_matrix = _ltb_month_matrix(scoring, pl.lit(run_index), -24, -1, "scoring")
    return {
        "train_history": train_matrix[:, :24],
        "train_future": train_matrix[:, 24:],
        "scoring_history": scoring_matrix,
        "scoring_keys": scoring["part_key"].unique().sort().to_list(),
        "train_parts": eligible.height,
        "train_excluded_incomplete": cohort.height - eligible.height,
        "decision_month": decision_month,
    }


In [ ]:
def local_parquet_path(value):
    path = Path(value).expanduser()
    if "://" in value or not path.is_absolute() or path.suffix.lower() != ".parquet":
        raise ValueError("로컬 Parquet 파일의 절대 경로를 입력하세요.")
    return path


if run_enabled:
    train_path = local_parquet_path(TRAIN_PATH)
    scoring_path = local_parquet_path(SCORING_PATH)
    output_path = local_parquet_path(OUTPUT_PATH) if OUTPUT_PATH else None
    if train_path.samefile(scoring_path):
        raise ValueError("학습·예측 입력은 서로 다른 파일이어야 합니다.")
    if output_path is not None:
        if output_path.exists() or output_path.is_symlink():
            raise FileExistsError("이미 있는 파일에 예측 결과를 덮어쓰지 않습니다.")
        if not output_path.parent.is_dir():
            raise FileNotFoundError("결과를 저장할 상위 폴더를 먼저 만들어 주세요.")

    inputs = prepare_ltb_operating_inputs(
        pl.read_parquet(train_path, columns=["part_key", "decision_month", "completed_month", "month", "demand_qty"]),
        pl.read_parquet(scoring_path, columns=["part_key", "month", "demand_qty"]),
        DECISION_MONTH,
    )
    print(f"기준월: {inputs['decision_month']}, 학습 후보: {inputs['train_parts']:,}개, "
          f"기간 조건으로 학습 제외: {inputs['train_excluded_incomplete']:,}개, "
          f"예측 대상: {len(inputs['scoring_keys']):,}개")


## 3. 예측 모델

**월별 패턴:** 학습 부품의 60개월 수요를 월별 비중으로 바꿔 K-Means로 묶습니다. Random Forest는 직전 24개월 이력만 보고 각 예측 대상에 적용할 패턴을 고릅니다. 60개월 실적 합계가 0인 부품은 월별 패턴 학습에서 제외합니다.

**60개월 총량:** 직전 24개월 월평균 × 60을 기준으로, 최근 6개월 수요 변화·24개월 추세·마지막 수요 이후 경과월을 반영한 부품별 계수를 적용합니다. 총량 학습에는 과거 수요가 있고 이후 60개월 수요가 0인 부품도 포함합니다. 최종 월별 예측량은 **총량 × 월별 비중**입니다.

In [ ]:
"""Fixed LTB forecasting pipeline: 24 observed months to 60 future months.

Callers supply point-in-time eligible, completed training rows. This module
does not access a data source, choose hyperparameters, or assess accuracy.
"""

import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import QuantileRegressor
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


LTB_OPERATING_LOOKBACK = 24
LTB_OPERATING_HORIZON = 60


def _ltb_operating_matrix(values, width, name):
    data = np.asarray(values, dtype=np.float64)
    if data.ndim != 2 or data.shape[1] != width:
        raise ValueError(f"{name} must have shape (parts, {width})")
    if not np.isfinite(data).all() or (data < 0).any():
        raise ValueError(f"{name} requires finite, nonnegative monthly demand")
    return data


def _ltb_operating_selector_features(history):
    total = history.sum(axis=1, keepdims=True)
    profile = np.divide(history, total, where=total > 0, out=np.zeros_like(history))
    recent = np.log1p(history[:, -6:].mean(axis=1))
    previous = np.log1p(history[:, -12:-6].mean(axis=1))
    return np.column_stack((
        profile, np.log1p(history.mean(axis=1)), recent, recent - previous,
        (history == 0).mean(axis=1),
    ))


def _ltb_operating_scale_features(history):
    month = np.arange(LTB_OPERATING_LOOKBACK, dtype=float) - 11.5
    recent = np.log1p(history[:, -6:].sum(axis=1))
    previous = np.log1p(history[:, -12:-6].sum(axis=1))
    slope = np.log1p(history) @ month / np.dot(month, month)
    last_demand = np.where(history > 0, np.arange(1, LTB_OPERATING_LOOKBACK + 1), 0).max(axis=1)
    return np.column_stack((recent - previous, slope, LTB_OPERATING_LOOKBACK - last_demand))


def fit_ltb_operating(train_history, train_future, *, k=3, min_cluster=20, seed=20261124):
    """Fit fixed K-Means profiles, a history-only RF, and a learned total scale.

    A positive future total is required to define a shape. A positive history
    total is required to fit a scale ratio; scale fitting keeps future zeros.
    The shape and scale populations are intentionally allowed to differ.
    """
    history = _ltb_operating_matrix(train_history, LTB_OPERATING_LOOKBACK, "train_history")
    future = _ltb_operating_matrix(train_future, LTB_OPERATING_HORIZON, "train_future")
    if len(history) != len(future) or k < 1 or min_cluster < 1:
        raise ValueError("fit needs aligned train rows and positive k/min_cluster")

    future_total = future.sum(axis=1)
    baseline = history.mean(axis=1) * LTB_OPERATING_HORIZON
    if not np.isfinite(future_total).all() or not np.isfinite(baseline).all():
        raise ValueError("training totals exceed supported numeric precision")
    shape_rows, scale_rows = future_total > 0, baseline > 0
    shapes = future[shape_rows] / future_total[shape_rows, None]
    if len(shapes) < k * min_cluster or len(np.unique(shapes, axis=0)) < k:
        raise ValueError(f"too few positive distinct shape profiles for {k} clusters")
    if not scale_rows.any():
        raise ValueError("no positive historical baseline is available for total calibration")

    scale_features = _ltb_operating_scale_features(history[scale_rows])
    with threadpool_limits(limits=2):
        labels = (
            KMeans(n_clusters=k, n_init=25, random_state=seed).fit_predict(shapes)
            if k > 1 else np.zeros(len(shapes), dtype=int)
        )
        counts = np.bincount(labels, minlength=k)
        if (counts < min_cluster).any():
            raise ValueError(f"cluster counts {counts.tolist()} fail minimum size {min_cluster}")
        centers = np.stack([shapes[labels == label].mean(axis=0) for label in range(k)])
        centers /= centers.sum(axis=1, keepdims=True)
        selector = (
            RandomForestClassifier(
                n_estimators=48, max_depth=6, min_samples_leaf=8, random_state=seed, n_jobs=2,
            ).fit(_ltb_operating_selector_features(history[shape_rows]), labels)
            if k > 1 else None
        )
        scaler = StandardScaler().fit(scale_features)
        scale_model = QuantileRegressor(quantile=0.5, alpha=0.01, solver="highs")
        scale_model.fit(
            scaler.transform(scale_features), future_total[scale_rows] / baseline[scale_rows],
            sample_weight=baseline[scale_rows],
        )
    return {
        "k": int(k), "cluster_counts": counts, "centers": centers, "selector": selector,
        "scaler": scaler, "scale_model": scale_model,
        "scale_feature_min": scale_features.min(axis=0),
        "scale_feature_max": scale_features.max(axis=0),
        "shape_training_parts": int(shape_rows.sum()),
        "scale_training_parts": int(scale_rows.sum()),
        "scale_zero_future_parts": int((scale_rows & ~shape_rows).sum()),
        "blend_strength": 1.0,
    }


def predict_ltb_operating(model, history):
    """Return monthly forecasts, 1-based cluster, and scale-input diagnostics.

    Zero demand in all 24 input months means that the forecast total is zero;
    the zero_history flag exposes this baseline limitation to the caller.
    """
    observed = _ltb_operating_matrix(history, LTB_OPERATING_LOOKBACK, "history")
    base = observed.mean(axis=1) * LTB_OPERATING_HORIZON
    feature = _ltb_operating_scale_features(observed)
    lower, upper = model["scale_feature_min"], model["scale_feature_max"]
    outside = ((feature < lower) | (feature > upper)).any(axis=1)
    with threadpool_limits(limits=2):
        labels = (
            model["selector"].predict(_ltb_operating_selector_features(observed))
            if len(observed) and model["selector"] is not None else np.zeros(len(observed), dtype=int)
        )
        factor = (
            np.maximum(model["scale_model"].predict(model["scaler"].transform(np.clip(feature, lower, upper))), 0.0)
            if len(observed) else np.empty(0, dtype=float)
        )
    share = model["centers"][labels]
    total = base * factor
    if not np.isfinite(total).all():
        raise ValueError("forecast totals exceed supported numeric precision")
    qty = total[:, None] * share
    np.testing.assert_allclose(qty.sum(axis=1), total, rtol=1e-12, atol=1e-10)
    return {
        "forecast_qty": qty, "forecast_total": total, "cluster": labels + 1,
        "monthly_share": share, "scale_factor": factor, "zero_history": base == 0,
        "feature_out_of_training_range": outside,
    }


## 4. 월별 예측 결과와 저장

`forecasts`는 부품별로 60행을 반환합니다. `forecast_qty`는 월별 예측량, `part_total_60m`은 해당 부품의 60개월 총량입니다. 여러 부품의 총량을 집계할 때는 행마다 반복되는 `part_total_60m` 대신 `forecast_qty`를 합산합니다. 군집 번호는 이번 실행 안에서만 해석합니다.

`review_required`가 참인 부품은 별도로 확인합니다. 직전 24개월 수요가 모두 0이면 모델의 계산상 총량도 0이지만, 앞으로 수요가 없다는 뜻은 아닙니다. 총량 보정에 쓰는 특성이 학습 범위를 벗어나면 보정 모델 입력에 한해 학습 범위로 제한하고 별도로 표시합니다. 결과 저장 시 기존 파일은 덮어쓰지 않습니다.

In [ ]:
def _shift_operating_month(value, offset):
    year, month = divmod(value.year * 12 + value.month - 1 + offset, 12)
    return date(year, month + 1, 1)


if run_enabled:
    model = fit_ltb_operating(
        inputs["train_history"], inputs["train_future"],
        k=K, min_cluster=MIN_CLUSTER_PARTS, seed=SEED,
    )
    prediction = predict_ltb_operating(model, inputs["scoring_history"])
    keys = inputs["scoring_keys"]
    as_of = inputs["decision_month"]
    horizon = LTB_OPERATING_HORIZON
    qty = prediction["forecast_qty"]
    total = prediction["forecast_total"]
    months = [_shift_operating_month(as_of, i) for i in range(horizon)]

    if qty.shape != (len(keys), horizon) or not np.isfinite(qty).all() or (qty < 0).any():
        raise ValueError("예측 수량의 행 수, 기간 또는 값이 잘못됐습니다.")
    np.testing.assert_allclose(qty.sum(axis=1), total, rtol=1e-12, atol=1e-10)
    zero_history = prediction["zero_history"]
    out_of_range = prediction["feature_out_of_training_range"]
    forecasts = pl.DataFrame({
        "decision_month": [as_of] * (len(keys) * horizon),
        "part_key": np.repeat(keys, horizon),
        "lead_month": np.tile(np.arange(1, horizon + 1), len(keys)),
        "forecast_month": months * len(keys),
        "forecast_qty": qty.ravel(),
        "part_total_60m": np.repeat(total, horizon),
        "cluster_id": np.repeat(prediction["cluster"], horizon),
        "monthly_share": prediction["monthly_share"].ravel(),
        "scale_factor": np.repeat(prediction["scale_factor"], horizon),
        "zero_history": np.repeat(zero_history, horizon),
        "feature_out_of_training_range": np.repeat(out_of_range, horizon),
        "review_required": np.repeat(zero_history | out_of_range, horizon),
    })
    if forecasts.select(pl.struct("part_key", "forecast_month").n_unique()).item() != forecasts.height:
        raise ValueError("부품별 예측 월이 중복됐습니다.")

    print(f"패턴 학습: {model['shape_training_parts']:,}개 "
          f"(군집별 {model['cluster_counts'].tolist()}), 총량 학습: {model['scale_training_parts']:,}개")
    print(f"예측: {forecasts.height:,}행 ({months[0]:%Y-%m}~{months[-1]:%Y-%m}), "
          f"검토 대상: {int((zero_history | out_of_range).sum()):,}개 부품")


In [ ]:
from tempfile import TemporaryDirectory

if run_enabled and output_path is not None:
    # 쓰기가 끝난 파일만 최종 경로에 추가하며, 이미 있는 파일은 덮어쓰지 않습니다.
    with TemporaryDirectory(prefix=".ltb-export-", dir=output_path.parent) as staging:
        temp_file = Path(staging) / "forecasts.parquet"
        forecasts.write_parquet(temp_file)
        temp_file.chmod(0o600)
        os.link(temp_file, output_path)
    print(f"Parquet 저장 완료: {forecasts.height:,}행")
